# Day 02：卷积层 —— 让机器自己学习特征

> 👁️ 第三周 · 视觉的征服 · 第 2 天

昨天我们用手工设计的卷积核（Sobel）检测了图像中的边缘。但现实世界中的图像千变万化——人脸、汽车、猫狗、文字……手工为每种特征设计卷积核是不可能的。

今天，我们要让卷积核**自己从数据中学习**。这就是 `nn.Conv2d`——一个可训练的卷积层。它的参数（卷积核的权重）不是手工设定的，而是通过反向传播从数据中学到的。

**今天的任务**：理解 `nn.Conv2d` 的工作原理，训练一个卷积核，亲眼看到它学到了什么特征。

---
## 1. 历史背景：从手工设计到自动学习

### 手工设计特征的黄金时代

在深度学习兴起之前（2000-2012），计算机视觉的主流方法是：

1. **手工设计特征提取器**：SIFT（1999）、HOG（2005）、SURF（2006）……
2. **用这些特征训练一个浅层分类器**：SVM、随机森林……

这就像让一个画家手工调配每一种颜料——费时费力，而且换一种画风就得重新调配。

### ImageNet 时刻：2012 年

2012 年，Alex Krizhevsky 带着 **AlexNet** 参加了 ImageNet 图像识别比赛。AlexNet 的核心创新不是架构多复杂，而是一个简单的想法：

**让卷积核自己学习该检测什么特征。**

AlexNet 有 5 个卷积层，每个卷积层有几十到几百个卷积核。这些卷积核的初始值是随机的，但经过训练后：
- 第一层学会了检测边缘、颜色、纹理
- 第二层学会了检测形状、图案
- 第三层学会了检测物体部件（眼睛、轮子……）
- 更深层学会了检测完整物体

AlexNet 以压倒性优势夺冠（错误率 15.3% vs 第二名 26.2%），标志着深度学习时代的正式开启。

### 可学习卷积核 vs 手工卷积核

| 对比维度 | 手工卷积核（Sobel） | 可学习卷积核（nn.Conv2d） |
|---------|-------------------|------------------------|
| 设计方式 | 人类专家手工设计 | 从数据中自动学习 |
| 适用场景 | 特定任务（边缘检测） | 任何任务（分类、检测、分割……） |
| 数量 | 几个到十几个 | 几十到几百个 |
| 泛化能力 | 固定不变 | 随任务自适应 |
| 代表方法 | Sobel、Canny、Gabor | AlexNet、VGG、ResNet |

可学习卷积核的本质是：**把卷积核的权重当作神经网络的参数，用反向传播来优化它们**。

---
## 2. 生活隐喻：从手工雕刻到 3D 打印

### 隐喻一：手工雕刻 vs 3D 打印

**手工设计卷积核**就像手工雕刻：
- 雕刻师（人类专家）根据经验，一刀一刀刻出形状
- 每换一种雕像，就得重新雕刻
- 费时费力，而且依赖雕刻师的技艺

**可学习卷积核**就像 3D 打印：
- 你只需要给机器看几张参考照片（训练数据）
- 机器自动「打印」出合适的形状（学习卷积核权重）
- 换一种雕像？换一批参考照片就行

3D 打印机不需要知道「雕刻原理」，它只需要数据和反馈（打印出来的像不像参考照片）。可学习卷积核也一样——它不需要知道「边缘检测的数学原理」，只需要数据和损失函数的反馈。

### 隐喻二：实习生成长记

想象你是一家公司的老板，需要一个人来「审核图片」（判断图片中是否有违规内容）。

**手工设计特征**的做法：
- 你写一本 500 页的审核手册：「如果左上角有红色且右下角有蓝色且……」
- 规则越来越复杂，总有漏网之鱼

**可学习卷积核**的做法：
- 你招一个实习生，给他看 10000 张「合规」和「违规」的图片
- 实习生自己摸索出判断规律
- 你不需要告诉他「看哪里」「看什么」——他自己学会

这个实习生的大脑就是卷积层——初始时什么都不会（随机权重），通过不断看样本、接受反馈（损失函数），逐渐学会该关注什么特征。

### 隐喻三：调鸡尾酒

卷积核的训练过程，就像调酒师摸索完美配方：

- **初始状态**：随机倒各种酒（随机初始化的卷积核）
- **尝一口**：和「目标味道」对比（前向传播 + 计算损失）
- **分析偏差**：太甜了？太酸了？（反向传播算梯度）
- **调整配方**：少放点糖浆，多放点柠檬汁（梯度下降更新权重）
- **重复**：直到调出完美味道（损失收敛）

每次调整都是微小的——你不会一次把糖浆全倒掉。学习率（learning rate）控制每次调整的幅度。

---
## 3. 数学直觉：nn.Conv2d 的内部构造

### nn.Conv2d 的参数

PyTorch 的 `nn.Conv2d` 有四个核心参数：

```python
nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0)
```

| 参数 | 含义 | 示例 |
|------|------|------|
| `in_channels` | 输入通道数（灰度图=1，RGB=3） | 1 |
| `out_channels` | 输出通道数（=卷积核个数） | 6 |
| `kernel_size` | 卷积核大小 | 3（表示 3×3） |
| `stride` | 滑动步长 | 1 |
| `padding` | 边缘补零圈数 | 1 |

### 卷积核的形状

一个 `nn.Conv2d(1, 6, 3)` 的卷积层，其内部权重形状是：

```
weight.shape = [6, 1, 3, 3]
               ↑  ↑  ↑  ↑
               │  │  └──┴── 卷积核空间尺寸 (3×3)
               │  └──────── 输入通道数 (1)
               └─────────── 输出通道数 (6，即 6 个卷积核)
```

偏置的形状是 `[6]`——每个输出通道一个偏置值。

### 多通道卷积

对于 RGB 彩色图像（3 通道），`nn.Conv2d(3, 16, 3)` 的权重形状是 `[16, 3, 3, 3]`。

每个输出通道的卷积核实际上是 `3×3×3` 的——它在**所有输入通道**上同时做卷积，然后把结果加起来。

```
输出通道 1 = 输入R ⊛ 核R + 输入G ⊛ 核G + 输入B ⊛ 核B + 偏置1
输出通道 2 = 输入R ⊛ 核R + 输入G ⊛ 核G + 输入B ⊛ 核B + 偏置2
...
```

这就是为什么 `in_channels` 必须和输入的通道数匹配。

### 前向传播的计算量

一个卷积层的前向传播计算量：

$$\text{FLOPs} = 2 \times k_h \times k_w \times C_{in} \times C_{out} \times H_{out} \times W_{out}$$

其中：
- $k_h \times k_w$ 是卷积核大小
- $C_{in}$ 是输入通道数
- $C_{out}$ 是输出通道数
- $H_{out} \times W_{out}$ 是输出特征图大小
- 乘以 2 是因为每次乘法对应一次加法（MAC = Multiply-ACcumulate）

例如：输入 28×28×1，卷积核 3×3，输出 6 通道，stride=1，padding=0：
- $H_{out} = 28 - 3 + 1 = 26$
- FLOPs = 2 × 3 × 3 × 1 × 6 × 26 × 26 = **73,008**

对比全连接层（784 → 128）：FLOPs = 2 × 784 × 128 = **200,704**

卷积不仅参数少，计算量也更小！

---
## 4. 代码实现：训练一个卷积核

下面用一个具体的任务来演示卷积核的学习过程：**训练一个卷积核，让它学会检测图像中的竖边**。

我们给网络看一些「有竖边」和「没有竖边」的简单图像，让它自己学会区分。

In [ ]:
# 设置中文字体
import matplotlib.pyplot as plt
try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

import torch
import torch.nn as nn

# 创建一个简单的卷积层：1个输入通道，1个输出通道，3x3卷积核
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, bias=False)

print("卷积层创建完成")
print(f"  输入通道: 1 (灰度图)")
print(f"  输出通道: 1 (一个卷积核)")
print(f"  卷积核大小: 3×3")
print(f"  权重形状: {conv.weight.shape}")
print(f"\n初始权重（随机）：")
print(conv.weight.data.squeeze())

### 构造训练数据

创建简单的训练数据：有竖边的图像（标签=1）和没有竖边的图像（标签=0）。

In [ ]:
import torch

# 构造训练数据：8 张 7×7 的简单图像
# 前 4 张有竖边（标签=1），后 4 张没有（标签=0）
X_train = torch.zeros(8, 1, 7, 7)  # [样本数, 通道, 高, 宽]

# 有竖边的图像：左边暗(0)，右边亮(1)
for i in range(4):
    X_train[i, 0, :, 3:] = 1.0  # 右边三列是亮的

# 没有竖边的图像：纯色或横边
X_train[4, 0] = 0.0  # 全黑
X_train[5, 0] = 1.0  # 全白
X_train[6, 0, 3:, :] = 1.0  # 横边（下边亮）
X_train[7, 0, :3, :] = 1.0  # 横边（上边亮）

y_train = torch.tensor([1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0])
y_train = y_train.view(-1, 1)  # 变成 [8, 1]

print("训练数据：")
for i in range(8):
    label = "有竖边" if y_train[i].item() > 0.5 else "无竖边"
    print(f"  样本 {i+1}: {label}")
    print(f"    {X_train[i, 0]}")
    print()

### 训练卷积核

用 MSE 损失和梯度下降来训练卷积核。观察卷积核的权重如何从随机逐渐变成类似 Sobel 的模式。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# 构造训练数据
X_train = torch.zeros(8, 1, 7, 7)
for i in range(4):
    X_train[i, 0, :, 3:] = 1.0
X_train[4, 0] = 0.0
X_train[5, 0] = 1.0
X_train[6, 0, 3:, :] = 1.0
X_train[7, 0, :3, :] = 1.0
y_train = torch.tensor([1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0]).view(-1, 1)

# 创建卷积层
conv = nn.Conv2d(1, 1, 3, bias=False)
optimizer = torch.optim.SGD(conv.parameters(), lr=0.01)  # SGD 优化器
criterion = nn.MSELoss()  # 均方误差损失

# 记录训练过程中的卷积核
kernel_history = []
kernel_history.append(conv.weight.data.clone())

print("开始训练卷积核...")
print("=" * 50)

epochs = 200
for epoch in range(epochs):
    # 前向传播：卷积 → 全局平均池化（取特征图的平均值作为输出）
    features = conv(X_train)  # [8, 1, 5, 5]
    output = features.mean(dim=[2, 3])  # [8, 1] 对空间维度取平均

    # 计算损失
    loss = criterion(output, y_train)

    # 反向传播
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # 每 50 轮记录一次卷积核
    if (epoch + 1) % 50 == 0:
        kernel_history.append(conv.weight.data.clone())
        print(f"Epoch {epoch+1:3d}: Loss = {loss.item():.4f}")

print("\n训练完成！")
print(f"最终损失: {loss.item():.4f}")

# 可视化卷积核的演变
fig, axes = plt.subplots(1, len(kernel_history), figsize=(14, 4))
titles = ['初始（随机）', '50轮', '100轮', '150轮', '200轮（最终）']
for i, (kernel, title) in enumerate(zip(kernel_history, titles)):
    im = axes[i].imshow(kernel[0, 0].numpy(), cmap='RdBu', vmin=-0.5, vmax=0.5)
    axes[i].set_title(title)
    axes[i].axis('off')
plt.suptitle('卷积核的演变：从随机到竖边检测器', fontsize=14)
plt.tight_layout()
plt.savefig('Week03/images/cnn_day02_learned_kernel.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n观察要点：")
print("  初始时卷积核是随机的（没有规律）")
print("  训练后，左边变成了负值（蓝色），右边变成了正值（红色）")
print("  这正是 Sobel-X 的模式——左边暗右边亮时输出大！")
print("  卷积核自己「发现」了竖边检测的规律！")

### 对比：训练前 vs 训练后

用训练前后的卷积核分别处理测试图像，直观对比效果。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# 构造训练数据并训练（同上）
X_train = torch.zeros(8, 1, 7, 7)
for i in range(4):
    X_train[i, 0, :, 3:] = 1.0
X_train[4, 0] = 0.0
X_train[5, 0] = 1.0
X_train[6, 0, 3:, :] = 1.0
X_train[7, 0, :3, :] = 1.0
y_train = torch.tensor([1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0]).view(-1, 1)

conv = nn.Conv2d(1, 1, 3, bias=False)
initial_kernel = conv.weight.data.clone()  # 保存初始卷积核

optimizer = torch.optim.SGD(conv.parameters(), lr=0.01)
criterion = nn.MSELoss()
for _ in range(200):
    features = conv(X_train)
    output = features.mean(dim=[2, 3])
    loss = criterion(output, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

trained_kernel = conv.weight.data.clone()  # 保存训练后的卷积核

# 创建测试图像
test_img = torch.zeros(1, 1, 7, 7)
test_img[0, 0, :, 3:] = 1.0  # 竖边图像

# 用初始卷积核处理
conv.weight.data = initial_kernel
before_features = conv(test_img)[0, 0].detach()

# 用训练后卷积核处理
conv.weight.data = trained_kernel
after_features = conv(test_img)[0, 0].detach()

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(test_img[0, 0].numpy(), cmap='gray')
axes[0].set_title('测试图像（竖边）')
axes[0].axis('off')

axes[1].imshow(before_features.numpy(), cmap='RdBu', vmin=-2, vmax=2)
axes[1].set_title('训练前的特征图（随机核）')
axes[1].axis('off')

axes[2].imshow(after_features.numpy(), cmap='RdBu', vmin=-2, vmax=2)
axes[2].set_title('训练后的特征图（学到的核）')
axes[2].axis('off')

plt.suptitle('训练前后卷积核的效果对比', fontsize=14)
plt.tight_layout()
plt.savefig('Week03/images/cnn_day02_before_training.png', dpi=150, bbox_inches='tight')
plt.show()

print("训练前：特征图杂乱无章，没有突出竖边")
print("训练后：特征图清晰显示了竖边的位置（正值=左边暗右边亮）")

---
## 5. 验证实验：多通道卷积核

真实 CNN 中，一个卷积层通常有多个输出通道（多个卷积核）。每个卷积核学习检测不同的特征。

下面创建一个有 4 个卷积核的卷积层，看看它们各自学到了什么。

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# 创建有 4 个卷积核的卷积层
conv_multi = nn.Conv2d(1, 4, 3, bias=False)
print("多卷积核卷积层：")
print(f"  输入通道: 1")
print(f"  输出通道: 4（4 个不同的卷积核）")
print(f"  权重形状: {conv_multi.weight.shape}")
print(f"  每个核独立学习不同的特征")

# 用 MNIST 的一张图片来演示
from torchvision import datasets, transforms

transform = transforms.Compose([transforms.ToTensor()])
mnist = datasets.MNIST(root='Week03/data', train=True, download=True, transform=transform)
img, label = mnist[0]  # 取第一张图片（数字 5）

# 前向传播
with torch.no_grad():
    features = conv_multi(img.unsqueeze(0))  # [1, 4, 26, 26]

# 可视化
fig, axes = plt.subplots(1, 5, figsize=(16, 4))

axes[0].imshow(img[0].numpy(), cmap='gray')
axes[0].set_title(f'原始图像（数字 {label}）')
axes[0].axis('off')

for i in range(4):
    axes[i+1].imshow(features[0, i].numpy(), cmap='RdBu')
    axes[i+1].set_title(f'卷积核 {i+1} 的特征图')
    axes[i+1].axis('off')

plt.suptitle('多个卷积核产生多个特征图（随机初始化，未训练）', fontsize=14)
plt.tight_layout()
plt.savefig('Week03/images/cnn_day02_test_images.png', dpi=150, bbox_inches='tight')
plt.show()

print("每个卷积核从不同「角度」观察图像")
print("训练后，不同核会学会检测不同的特征（边缘、纹理、颜色……）")
print("这就是 CNN 的「多视角」能力")

---
## 翻译词典

| 生活直觉 | 深度学习术语 |
|----------|-------------|
| 自动学习「模板」的内容 | 可学习卷积核（nn.Conv2d） |
| 卷积后的「结果图」 | 特征图（Feature Map） |
| 多个模板同时检测 | 多通道卷积 |
| 输出尺寸公式 | padding、stride、kernel_size |
| 实习生自己摸索规律 | 从数据中学习特征 |
| 调鸡尾酒的每次微调 | 梯度下降更新权重 |
| 3D 打印参考照片 | 训练数据 |

---
## 今日总结

| 概念 | 直觉理解 |
|------|----------|
| nn.Conv2d | 一个可学习的卷积层，自动从数据中提取特征 |
| 前向传播 | 图片通过卷积核，得到特征图 |
| 训练 | 调整卷积核的权重，让输出接近目标 |
| 卷积核可视化 | 学到的权重模式，揭示网络关注什么特征 |
| 多通道 | 多个卷积核并行工作，从不同角度观察图像 |

**关键洞察**：
- 全连接层：每个像素独立连接 → 参数爆炸
- 卷积层：局部连接 + 权值共享 → 参数极少
- 训练的本质：让卷积核学会「检测有用的特征」
- 不需要告诉它检测什么——它自己从数据中发现

**明日预告**：卷积层产生了大量特征图，但很多信息是冗余的。明天学习**池化（Pooling）**——如何压缩特征图，在保持关键信息的同时减少计算量。